# Identify Problematic Column in migration_event_log table

Error: `Could not convert '+56901-11-11 18:39:56.898' into the associated python type`

Then fetch the data in a csv

In [1]:
import subprocess as sp
from io import StringIO
import pandas as pd

def fetch_data(sql_query):
    print(f"Executing query...")
    try:
        data = pd.read_csv(
            StringIO(
                sp.check_output(
                    f'''pharos sql run --sql "{sql_query}" | jq -r '.result.data' ''',
                    shell=True,
                ).decode("utf-8")
            )
        )
        print(f"Success! Rows: {len(data)}")
        return data
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
# Real Culprits of the Timeout Error: time_start and time_end columns
query = "SELECT time_start FROM swh.migration_event_log WHERE DATE(wd_event_date) <= CURRENT_DATE"
query = "SELECT time_end FROM swh.migration_event_log WHERE DATE(wd_event_date) <= CURRENT_DATE"

In [ ]:
# Fix for the error
count_query = """
SELECT COUNT(*) as row_count
FROM swh.migration_event_log 
WHERE DATE(wd_event_date) <= CURRENT_DATE
  AND DATE(wd_event_date) >= DATE('2023-01-01')
  AND time_start <= CURRENT_TIMESTAMP
  AND time_end <= CURRENT_TIMESTAMP
"""
result = fetch_data(count_query)
if result is not None:
  print(f"Total rows matching filter: {result['row_count']}")

In [ ]:
# Fetch data in configurable chunks to avoid timeout because the table is huge
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

def fetch_in_chunks(start_date, end_date, chunk_size='yearly'):
    """
    Fetch migration_event_log data in chunks.
    
    Args:
        start_date: Start date string 'YYYY-MM-DD'
        end_date: End date string 'YYYY-MM-DD'
        chunk_size: 'yearly', 'half-yearly', 'quarterly', or 'monthly'
    """
    chunk_map = {
        'yearly': relativedelta(years=1),
        'half-yearly': relativedelta(months=6),
        'quarterly': relativedelta(months=3),
        'monthly': relativedelta(months=1),
    }
    
    if chunk_size not in chunk_map:
        raise ValueError(f"chunk_size must be one of: {list(chunk_map.keys())}")
    
    delta = chunk_map[chunk_size]
    all_data = []
    current_start = datetime.strptime(start_date, '%Y-%m-%d')
    final_end = datetime.strptime(end_date, '%Y-%m-%d')
    
    print(f"Fetching data in {chunk_size} chunks from {start_date} to {end_date}\n")
    
    while current_start < final_end:
        current_end = min(current_start + delta, final_end)
        
        query = f"""
        SELECT * 
        FROM swh.migration_event_log 
        WHERE DATE(wd_event_date) >= DATE('{current_start.strftime('%Y-%m-%d')}')
          AND DATE(wd_event_date) < DATE('{current_end.strftime('%Y-%m-%d')}')
          AND time_start <= CURRENT_TIMESTAMP
          AND time_end <= CURRENT_TIMESTAMP
        """
        
        date_label = f"{current_start.strftime('%Y-%m-%d')} to {current_end.strftime('%Y-%m-%d')}"
        print(f"Fetching {date_label}...", end=" ")
        result = fetch_data(query)
        
        if result is not None and len(result) > 0:
            all_data.append(result)
            print(f"Got {len(result):,} rows")
        else:
            print("No data or error")
        
        current_start = current_end
    
    if all_data:
        combined = pd.concat(all_data, ignore_index=True)
        print(f"\n✅ Total combined rows: {len(combined):,}")
        return combined
    return None

In [3]:
# Initialize list to collect all data chunks
all_chunks = []

In [ ]:
# Part 1: Sparse data period (2023-01-01 to 2024-08-01) - single query
print("=" * 60)
print("Part 1: Sparse period (2023-01-01 to 2024-08-01)")
print("=" * 60)
query_sparse = """
SELECT * 
FROM swh.migration_event_log 
WHERE DATE(wd_event_date) >= DATE('2023-01-01')
  AND DATE(wd_event_date) < DATE('2024-08-01')
  AND time_start <= CURRENT_TIMESTAMP
  AND time_end <= CURRENT_TIMESTAMP
"""
sparse_data = fetch_data(query_sparse)
if sparse_data is not None and len(sparse_data) > 0:
    all_chunks.append(sparse_data)
    print(f"✅ Got {len(sparse_data):,} rows from sparse period")

Part 1: Sparse period (2023-01-01 to 2024-07-01)
Executing query...


2026-01-15 08:45:02,115 INFO executing SQL


SELECT * 
FROM swh.migration_event_log 
WHERE DATE(wd_event_date) >= DATE('2023-01-01')
  AND DATE(wd_event_date) < DATE('2024-07-01')
  AND time_start <= CURRENT_TIMESTAMP
  AND time_end <= CURRENT_TIMESTAMP




Success! Rows: 10
✅ Got 10 rows from sparse period


[====================================================================] FINISHED
2026-01-15 08:45:08,897 INFO SQL completed


In [5]:
# Part 2: Heavy data period (2024-08-01 onwards) - month by month
print("\n" + "=" * 60)
print("Part 2: Heavy period (2024-08-01 to present) - Monthly")
print("=" * 60)

monthly_data = fetch_in_chunks('2024-08-01', datetime.now().strftime('%Y-%m-%d'), chunk_size='monthly')
if monthly_data is not None and len(monthly_data) > 0:
    all_chunks.append(monthly_data)


Part 2: Heavy period (2024-07-01 to present) - Monthly
Fetching data in monthly chunks from 2024-07-01 to 2026-01-15

Fetching 2024-07-01 to 2024-08-01... Executing query...


2026-01-15 08:45:16,461 INFO executing SQL


        SELECT * 
        FROM swh.migration_event_log 
        WHERE DATE(wd_event_date) >= DATE('2024-07-01')
          AND DATE(wd_event_date) < DATE('2024-08-01')
          AND time_start <= CURRENT_TIMESTAMP
          AND time_end <= CURRENT_TIMESTAMP
        

[====================================================================] FINISHED
2026-01-15 08:45:20,212 INFO SQL completed


Success! Rows: 40
Got 40 rows
Fetching 2024-08-01 to 2024-09-01... Executing query...


2026-01-15 08:45:20,863 INFO executing SQL


        SELECT * 
        FROM swh.migration_event_log 
        WHERE DATE(wd_event_date) >= DATE('2024-08-01')
          AND DATE(wd_event_date) < DATE('2024-09-01')
          AND time_start <= CURRENT_TIMESTAMP
          AND time_end <= CURRENT_TIMESTAMP
        

[====================================================================] FINISHED
2026-01-15 08:46:55,731 INFO SQL completed
/var/folders/s6/xr135lb50s954wv51r1cvx4w0000gp/T/ipykernel_62525/82383317.py:8: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(


Success! Rows: 836915
Got 836,915 rows
Fetching 2024-09-01 to 2024-10-01... Executing query...


...

Success! Rows: 1491021
Got 1,491,021 rows
Fetching 2025-12-01 to 2026-01-01... Executing query...


2026-01-15 09:24:32,695 INFO executing SQL


        SELECT * 
        FROM swh.migration_event_log 
        WHERE DATE(wd_event_date) >= DATE('2025-12-01')
          AND DATE(wd_event_date) < DATE('2026-01-01')
          AND time_start <= CURRENT_TIMESTAMP
          AND time_end <= CURRENT_TIMESTAMP
        

[====================================================================] FINISHED
2026-01-15 09:27:57,423 INFO SQL completed
/var/folders/s6/xr135lb50s954wv51r1cvx4w0000gp/T/ipykernel_62525/82383317.py:8: DtypeWarning: Columns (44,45,46,49,50,51,52,53,54,55,56,57,58,60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(


Success! Rows: 1548891
Got 1,548,891 rows
Fetching 2026-01-01 to 2026-01-15... Executing query...


2026-01-15 09:28:42,998 INFO executing SQL


        SELECT * 
        FROM swh.migration_event_log 
        WHERE DATE(wd_event_date) >= DATE('2026-01-01')
          AND DATE(wd_event_date) < DATE('2026-01-15')
          AND time_start <= CURRENT_TIMESTAMP
          AND time_end <= CURRENT_TIMESTAMP
        

[====================================================================] FINISHED
2026-01-15 09:29:41,767 INFO SQL completed
/var/folders/s6/xr135lb50s954wv51r1cvx4w0000gp/T/ipykernel_62525/82383317.py:8: DtypeWarning: Columns (4,5,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(


Success! Rows: 592470
Got 592,470 rows

✅ Total combined rows: 18,103,139


In [6]:
# Combine all chunks and save
if all_chunks:
    combined_data = pd.concat(all_chunks, ignore_index=True)
    print("\n" + "=" * 60)
    print(f"✅ TOTAL ROWS COLLECTED: {len(combined_data):,}")
    print("=" * 60)
else:
    combined_data = None
    print("❌ No data collected")


✅ TOTAL ROWS COLLECTED: 18,103,149


In [8]:
# Save the combined data
if combined_data is not None:
    combined_data.to_csv("migration_event_log.csv", index=False)
    print(f"Saved {len(combined_data):,} rows to csv/migration_event_log.csv")

Saved 18,103,149 rows to csv/migration_event_log.csv


In [ ]:
## Fallback (Optional): If a specific month times out, break it into weeks
def fetch_single_range(start_date, end_date):
    """Fetch a single date range."""
    query = f"""
    SELECT * 
    FROM swh.migration_event_log 
    WHERE DATE(wd_event_date) >= DATE('{start_date}')
      AND DATE(wd_event_date) < DATE('{end_date}')
      AND time_start <= CURRENT_TIMESTAMP
      AND time_end <= CURRENT_TIMESTAMP
    """
    print(f"Fetching {start_date} to {end_date}...", end=" ")
    result = fetch_data(query)
    if result is not None:
        print(f"Got {len(result):,} rows")
    return result

def fetch_by_week(year, month):
    """Fetch a specific month week by week."""
    from calendar import monthrange
    
    start = datetime(year, month, 1)
    _, last_day = monthrange(year, month)
    end = datetime(year, month, last_day) + timedelta(days=1)
    
    print(f"\nBreaking down {year}-{month:02d} into weekly chunks:")
    weekly_data = []
    current = start
    
    while current < end:
        week_end = min(current + timedelta(days=7), end)
        result = fetch_single_range(
            current.strftime('%Y-%m-%d'), 
            week_end.strftime('%Y-%m-%d')
        )
        if result is not None and len(result) > 0:
            weekly_data.append(result)
        current = week_end
    
    if weekly_data:
        combined = pd.concat(weekly_data, ignore_index=True)
        print(f"✅ Month {year}-{month:02d} total: {len(combined):,} rows")
        return combined
    return None

# Example: If August 2024 times out, run this:
# aug_2024_data = fetch_by_week(2024, 8)
# all_chunks.append(aug_2024_data)